In [ ]:
!pip install rdkit
!pip install tensorflow

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
import random as rn
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow import keras
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.utils import Sequence

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
%matplotlib inline

%tensorflow_version 2.x
import tensorflow as tf

In [3]:

import os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers


In [ ]:

SEED = 42
N_SPLITS = 5
EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 1e-3

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)
print("TensorFlow version:", tf.__version__)


In [ ]:

# Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
print("Dataset shape:", df.shape)
display(df.head())

target_col = "HIV_active"
y = df[target_col].values

# Keep only numeric predictor columns
X_df = df.drop(columns=[target_col]).select_dtypes(include=["float64", "int64", "float32", "int32"]).copy()
print("Numeric feature shape:", X_df.shape)

X = X_df.values.astype("float32")
n_features = X.shape[1]


In [6]:

def build_brnn_model(input_length):
    model = models.Sequential([
        layers.Input(shape=(input_length, 1)),
        layers.Bidirectional(
            layers.SimpleRNN(
                64,
                return_sequences=True,
                dropout=0.2,
                recurrent_dropout=0.0,
                kernel_regularizer=regularizers.l2(1e-4)
            )
        ),
        layers.Bidirectional(
            layers.SimpleRNN(
                32,
                return_sequences=False,
                dropout=0.2,
                recurrent_dropout=0.0,
                kernel_regularizer=regularizers.l2(1e-4)
            )
        ),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model


In [7]:

def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype("int32")
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

def prepare_fold_data(X_train_full, X_test, y_train_full, y_test, random_state):
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.2,
        random_state=random_state,
        stratify=y_train_full
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    return X_train, X_val, X_test, y_train, y_val, y_test


In [8]:

def run_brnn_5fold_cv(X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    fold_results = []
    histories = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        print("\n" + "="*28 + f" FOLD {fold} " + "="*28)
        set_seed(seed + fold)

        X_train_full, X_test = X[train_idx], X[test_idx]
        y_train_full, y_test = y[train_idx], y[test_idx]

        X_train, X_val, X_test, y_train, y_val, y_test = prepare_fold_data(
            X_train_full, X_test, y_train_full, y_test, random_state=seed + fold
        )

        model = build_brnn_model(input_length=X_train.shape[1])

        early_stop = callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1
        )

        reduce_lr = callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-5,
            verbose=1
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )
        histories.append(history.history)

        y_val_prob = model.predict(X_val, verbose=0).ravel()
        y_test_prob = model.predict(X_test, verbose=0).ravel()

        val_metrics = evaluate_predictions(y_val, y_val_prob, threshold=0.5)
        test_metrics = evaluate_predictions(y_test, y_test_prob, threshold=0.5)

        row = {
            "fold": fold,
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_f1": test_metrics["f1"],
            "test_roc_auc": test_metrics["roc_auc"],
            "epochs_trained": len(history.history["loss"])
        }
        fold_results.append(row)

        print("Validation:",
              {k.replace("val_", ""): round(v, 4) for k, v in row.items() if k.startswith("val_")})
        print("Test:",
              {k.replace("test_", ""): round(v, 4) for k, v in row.items() if k.startswith("test_")})
        print("Epochs trained:", row["epochs_trained"])

    results_df = pd.DataFrame(fold_results)

    metric_cols = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]
    summary_rows = []
    for col in metric_cols:
        mean_val = results_df[col].mean()
        std_val = results_df[col].std(ddof=1)
        var_val = results_df[col].var(ddof=1)
        summary_rows.append({
            "Metric": col.replace("test_", "").upper(),
            "Mean": mean_val,
            "Std": std_val,
            "Variance": var_val,
            "Formatted": f"{mean_val:.4f} ± {std_val:.4f}"
        })

    summary_df = pd.DataFrame(summary_rows)
    return results_df, summary_df, histories


In [ ]:

results_df, summary_df, histories = run_brnn_5fold_cv(X, y, n_splits=N_SPLITS, seed=SEED)

print("\nFold-wise results:")
display(results_df)

print("\nSummary results:")
display(summary_df)

results_df.to_csv("BRNN_fold_results.csv", index=False)
summary_df.to_csv("BRNN_summary_results.csv", index=False)

print("Saved: BRNN_fold_results.csv")
print("Saved: BRNN_summary_results.csv")


In [10]:

def plot_average_history(histories):
    max_len = max(len(h["loss"]) for h in histories)

    train_loss_mat, val_loss_mat = [], []
    train_acc_mat, val_acc_mat = [], []

    for h in histories:
        tl = np.array(h["loss"], dtype=float)
        vl = np.array(h["val_loss"], dtype=float)
        ta = np.array(h["accuracy"], dtype=float)
        va = np.array(h["val_accuracy"], dtype=float)

        def pad(arr, length):
            if len(arr) < length:
                arr = np.pad(arr, (0, length-len(arr)), mode="edge")
            return arr

        train_loss_mat.append(pad(tl, max_len))
        val_loss_mat.append(pad(vl, max_len))
        train_acc_mat.append(pad(ta, max_len))
        val_acc_mat.append(pad(va, max_len))

    avg_train_loss = np.mean(train_loss_mat, axis=0)
    avg_val_loss = np.mean(val_loss_mat, axis=0)
    avg_train_acc = np.mean(train_acc_mat, axis=0)
    avg_val_acc = np.mean(val_acc_mat, axis=0)

    epochs = np.arange(1, max_len + 1)

    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_acc, label="Train Accuracy")
    plt.plot(epochs, avg_val_acc, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("BRNN Average Accuracy Across 5 Folds")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_loss, label="Train Loss")
    plt.plot(epochs, avg_val_loss, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("BRNN Average Loss Across 5 Folds")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
plot_average_history(histories)

## BRNN docking preparation block

This block is prepared for Colab.  
First RDKit is installed, then the best fold is rebuilt and:

- top 10 candidates
- shared column format
- final 2 candidates
- `.smi` file
- 2D molecule drawing

are generated.

In [ ]:
# Colab RDKit install
import sys, subprocess, pkgutil

if pkgutil.find_loader("rdkit") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"])

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

print("RDKit OK")

In [ ]:
# ================================
# BRNN FINAL PIPELINE (COLAB VERSION)
# ================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 1) Select the best fold
best_fold = int(results_df["test_roc_auc"].astype(float).idxmax()) + 1
print(f"Using best fold: {best_fold}")

# 2) SMILES column
if "smiles" not in df.columns:
    raise ValueError("The dataset does not contain a 'smiles' column.")
smiles_col = "smiles"

# 3) Rebuild the same CV split
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(skf.split(X, y))
train_idx, test_idx = splits[best_fold - 1]

X_train_full, X_test_raw = X[train_idx], X[test_idx]
y_train_full, y_test_raw = y[train_idx], y[test_idx]
smiles_test_full = df.iloc[test_idx][smiles_col].reset_index(drop=True)

# 4) Rebuild the same preprocessing split
X_train, X_val, X_test, y_train, y_val, y_test = prepare_fold_data(
    X_train_full, X_test_raw, y_train_full, y_test_raw, random_state=SEED + best_fold
)

# prepare_fold_data splits validation with train_test_split.
# Because the test set is unchanged, smiles_test_full is directly aligned with X_test_raw / y_test_raw.
smiles_test = smiles_test_full.iloc[:len(y_test)].reset_index(drop=True)

# 5) Rebuild and train the model
set_seed(SEED + best_fold)

model = build_brnn_model(input_length=X_train.shape[1])

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

# 6) Generate predictions
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

df_pred = pd.DataFrame({
    "fold": best_fold,
    "smiles": smiles_test,
    "y_true": np.array(y_test),
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 7) Top 10 candidates
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 8) Descriptor hesaplama
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

desc_rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        desc_rows.append(d)

df_desc = pd.DataFrame(desc_rows)

# 9) Shared column order
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nBRNN TOP 10 CANDIDATES:")
display(df_desc)

# 10) Final 2 candidates
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nBRNN FINAL 2 CANDIDATES:")
display(final_df)

# 11) Save
df_desc.to_csv("BRNN_top_10_candidates.csv", index=False)
final_df.to_csv("BRNN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("BRNN_docking_input.smi", index=False, header=False)

print("\nSaved: BRNN_top_10_candidates.csv")
print("Saved: BRNN_final_2_candidates.csv")
print("Saved: BRNN_docking_input.smi")

# 12) Draw the best 2 molecules
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(320, 320),
    legends=[
        f"BRNN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"BRNN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)

In [ ]:
# ================================
# BRNN FINAL PIPELINE (RDKIT-SAFE)
# ================================

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

# -------------------------------------------------
# 1) RDKit check
# -------------------------------------------------
RDKIT_AVAILABLE = True
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Draw
    from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED
except Exception:
    RDKIT_AVAILABLE = False
    print("RDKit not available. Fallback descriptor and text-visual mode will be used.")

# -------------------------------------------------
# 2) Fallback descriptor
# -------------------------------------------------
def compute_desc_fallback(smiles):
    if not isinstance(smiles, str):
        return None

    atoms = re.findall(r'[A-Z][a-z]?', smiles)

    mw_table = {
        'C': 12.011, 'H': 1.008, 'O': 15.999, 'N': 14.007,
        'S': 32.06, 'P': 30.974, 'F': 18.998, 'Cl': 35.45,
        'Br': 79.904, 'I': 126.90, 'Se': 78.971
    }

    mw = sum(mw_table.get(a, 0) for a in atoms)
    hbd = smiles.count('N') + smiles.count('O')
    hba = hbd
    logp = smiles.count('C') * 0.54 - smiles.count('O') * 1.5 - smiles.count('N') * 1.0
    tpsa = hba * 12.0
    rot = smiles.count('(')

    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    qed = min(1.0, max(0.0, 0.6 * (1 / (1 + abs(logp))) + 0.4 * (1 / (1 + hbd))))

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

def compute_desc(smiles):
    if RDKIT_AVAILABLE:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        mw = Descriptors.MolWt(mol)
        logp = Crippen.MolLogP(mol)
        hbd = RdLipinski.NumHDonors(mol)
        hba = RdLipinski.NumHAcceptors(mol)
        tpsa = Descriptors.TPSA(mol)
        rot = RdLipinski.NumRotatableBonds(mol)
        qed = QED.qed(mol)
        lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

        return {
            "MW": mw,
            "LogP": logp,
            "HBD": hbd,
            "HBA": hba,
            "TPSA": tpsa,
            "RotatableBonds": rot,
            "QED": qed,
            "Lipinski": lip
        }
    else:
        return compute_desc_fallback(smiles)

# -------------------------------------------------
# 3) Select the best fold
# -------------------------------------------------
best_fold = int(results_df["test_roc_auc"].astype(float).idxmax()) + 1
print(f"Using best fold: {best_fold}")

# -------------------------------------------------
# 4) SMILES column
# -------------------------------------------------
possible_smiles_cols = [c for c in df.columns if "smile" in c.lower()]
if len(possible_smiles_cols) == 0:
    raise ValueError("SMILES column not found in df.")
smiles_col = possible_smiles_cols[0]

# -------------------------------------------------
# 5) Rebuild the same split
# -------------------------------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(skf.split(X, y))
train_idx, test_idx = splits[best_fold - 1]

X_train_full, X_test_raw = X[train_idx], X[test_idx]
y_train_full, y_test_raw = y[train_idx], y[test_idx]
smiles_test = df.iloc[test_idx][smiles_col].reset_index(drop=True)

# -------------------------------------------------
# 6) Preprocessing
# -------------------------------------------------
X_train, X_val, X_test, y_train, y_val, y_test = prepare_fold_data(
    X_train_full, X_test_raw, y_train_full, y_test_raw, random_state=SEED + best_fold
)

if len(smiles_test) != len(y_test):
    smiles_test = smiles_test.iloc[:len(y_test)].reset_index(drop=True)

# -------------------------------------------------
# 7) Rebuild and train the model
# -------------------------------------------------
set_seed(SEED + best_fold)

model = build_brnn_model(input_length=X_train.shape[1])

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

# -------------------------------------------------
# 8) Prediction
# -------------------------------------------------
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

df_pred = pd.DataFrame({
    "fold": best_fold,
    "smiles": smiles_test,
    "y_true": np.array(y_test),
    "y_pred": y_pred,
    "y_prob": y_prob
})

# -------------------------------------------------
# 9) Top 10
# -------------------------------------------------
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nBRNN TOP 10 CANDIDATES:")
display(df_desc)

# -------------------------------------------------
# 10) Final 2
# -------------------------------------------------
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nBRNN FINAL 2 CANDIDATES:")
display(final_df)

# -------------------------------------------------
# 11) Save
# -------------------------------------------------
df_desc.to_csv("BRNN_top_10_candidates.csv", index=False)
final_df.to_csv("BRNN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("BRNN_docking_input.smi", index=False, header=False)

print("\nSaved: BRNN_top_10_candidates.csv")
print("Saved: BRNN_final_2_candidates.csv")
print("Saved: BRNN_docking_input.smi")

# -------------------------------------------------
# 12) Draw
# -------------------------------------------------
if RDKIT_AVAILABLE:
    mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=2,
        subImgSize=(320, 320),
        legends=[
            f"BRNN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
            f"BRNN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
        ]
    )
    display(img)
else:
    fig, axes = plt.subplots(1, min(2, len(final_df)), figsize=(12, 3))
    if len(final_df) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, final_df.iterrows()):
        ax.axis("off")
        ax.text(
            0.02, 0.5,
            f"SMILES:\n{row['smiles']}\n\nProb={row['y_prob']:.3f}",
            fontsize=10,
            va="center",
            wrap=True,
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="black")
        )
    plt.tight_layout()
    plt.show()

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display

mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

drawer = rdMolDraw2D.MolDraw2DSVG(800, 400, 400, 400)
drawer.DrawMolecules(
    mols,
    legends=[
        f"BRNN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"BRNN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)
drawer.FinishDrawing()
svg = drawer.GetDrawingText()
display(SVG(svg))